# M2.2 · Naive Bayes

Sayarak öğrenen sınıflandırıcı. Altı satırlık bir tabloda posterior'u Laplace düzeltmesiyle ve düzeltmesiz elle hesaplıyor; saf Python'da kategorik Naive Bayes yazıyor; NumPy'da Gaussian NB'yi, pandas'ta olasılık tablolarını döngüsüz kuruyor; PyTorch'ta metin için multinomial NB eğitiyor ve en sonda 20 farklı α değerini tek tensörle değerlendiriyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m2-2-naive-bayes.html](https://m-gonul.github.io/pytorch-lab/m2-2-naive-bayes.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m2_2_naive_bayes.py
CHECKS = {
 "b": "import math\nimport random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef ref_proba(rows, labels, alpha, q):\n    classes = sorted(set(labels))\n    d = len(rows[0])\n    V = [len({r[j] for r in rows}) for j in range(d)]\n    sc = {}\n    for c in classes:\n        sub = [r for r, l in zip(rows, labels) if l == c]\n        s = len(sub) / len(rows)\n        for j in range(d):\n            s *= (sum(r[j] == q[j] for r in sub) + alpha) / (len(sub) + alpha * V[j])\n        sc[c] = s\n    z = sum(sc.values())\n    return {c: v / z for c, v in sc.items()}\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch/pandas kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\", \"pandas\", \"sklearn\"}))\n    rows = [(\"g\", \"z\"), (\"g\", \"s\"), (\"y\", \"s\"), (\"y\", \"z\"), (\"g\", \"s\"), (\"y\", \"s\")]\n    labels = [\"E\", \"E\", \"H\", \"E\", \"H\", \"H\"]\n\n    def hand():\n        p0 = need(ns, \"predict_proba\")(need(ns, \"fit_nb\")(rows, labels, alpha=0.0), (\"g\", \"s\"))\n        p1 = need(ns, \"predict_proba\")(need(ns, \"fit_nb\")(rows, labels, alpha=1.0), (\"g\", \"s\"))\n        close([p0[\"E\"], p0[\"H\"]], [0.4, 0.6], what=\"α = 0\")\n        close([p1[\"E\"], p1[\"H\"]], [3 / 7, 4 / 7], what=\"α = 1\")\n    R.test(\"elle örnek: 0.4 (α = 0) ve 0.4286 (α = 1)\", hand)\n\n    def zero():\n        p = need(ns, \"predict_proba\")(need(ns, \"fit_nb\")(rows, labels, alpha=0.0), (\"y\", \"z\"))\n        close([p[\"E\"], p[\"H\"]], [1.0, 0.0], what=\"(yağmur, zayıf), α = 0\")\n        p1 = need(ns, \"predict_proba\")(need(ns, \"fit_nb\")(rows, labels, alpha=1.0), (\"y\", \"z\"))\n        ensure(0 < p1[\"H\"] < 1, \"Laplace ile H olasılığı 0 ile 1 arasında olmalı\")\n        return f\"α = 0: P(H) = 0; α = 1: P(H) = {p1['H']:.3f}\"\n    R.test(\"sıfır sayım ve Laplace\", zero)\n\n    def rnd():\n        rng = random.Random(0)\n        rows2 = [tuple(rng.choice(\"abc\") for _ in range(4)) for _ in range(60)]\n        labs = [(\"x\" if r.count(\"a\") >= 2 else \"y\") if rng.random() < 0.9 else rng.choice(\"xy\") for r in rows2]\n        m = need(ns, \"fit_nb\")(rows2, labs, alpha=0.5)\n        for q in rows2[:10]:\n            got = need(ns, \"predict_proba\")(m, q)\n            want = ref_proba(rows2, labs, 0.5, q)\n            close([got[c] for c in sorted(want)], [want[c] for c in sorted(want)], rtol=1e-9)\n            ensure(need(ns, \"predict\")(m, q) == max(want, key=want.get), \"predict argmax değil\")\n    R.test(\"rastgele veri: referansla aynı, predict = argmax\", rnd)\n\n    def underflow():\n        rows3 = [tuple(\"ab\"[(i >> k) & 1] for k in range(400)) for i in range(8)]\n        labs3 = [\"p\", \"q\"] * 4\n        p = need(ns, \"predict_proba\")(need(ns, \"fit_nb\")(rows3, labs3, alpha=1.0), rows3[0])\n        ensure(all(math.isfinite(v) for v in p.values()) and abs(sum(p.values()) - 1) < 1e-9, \"400 özellikte olasılıklar bozuldu; log uzayında çalış\")\n    R.test(\"400 özellikte alt taşma yok (log uzayı)\", underflow)\n",
 "c": "import numpy as np\nimport pandas as pd\nfrom labcheck import close, no_loops, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    R.test(\"torch/sklearn kullanılmıyor\", lambda: no_imports(src, {\"torch\", \"sklearn\"}))\n    rng = np.random.default_rng(4)\n    C, d = 3, 4\n    y = rng.integers(0, C, 150)\n    X = rng.normal(size=(150, d)) * (1 + y[:, None]) + y[:, None] * 1.5\n\n    def fit():\n        lp, mu, var = need(ns, \"fit_gnb\")(X, y, C)\n        for c in range(C):\n            close(mu[c], X[y == c].mean(0), what=f\"μ sınıf {c}\")\n            close(var[c], X[y == c].var(0), rtol=1e-6, atol=1e-8, what=f\"σ² sınıf {c}\")\n        close(np.exp(lp), np.bincount(y) / 150, what=\"prior\")\n    R.test(\"fit_gnb: sınıf ortalaması, MLE varyansı, prior\", fit)\n\n    def pred():\n        m = need(ns, \"fit_gnb\")(X, y, C)\n        L = need(ns, \"predict_log_proba_gnb\")(m, X)\n        ensure(np.asarray(L).shape == (150, C), f\"shape {np.asarray(L).shape}\")\n        close(np.exp(L).sum(1), np.ones(150), what=\"olasılık toplamı\")\n        lp, mu, var = m\n        ref = np.stack([lp[c] - 0.5 * (np.log(2 * np.pi * var[c]) + (X - mu[c]) ** 2 / var[c]).sum(1) for c in range(C)], 1)\n        ref -= np.log(np.exp(ref - ref.max(1, keepdims=True)).sum(1, keepdims=True)) + ref.max(1, keepdims=True)\n        close(L, ref, rtol=1e-7, atol=1e-9)\n        return f\"eğitim doğruluğu {(np.argmax(L, 1) == y).mean():.3f}\"\n    R.test(\"predict_log_proba_gnb normalize ve doğru\", pred)\n\n    def tab():\n        df = pd.DataFrame({\"hava\": list(\"ggyygy\"), \"oyna\": list(\"EEHEHH\")})\n        t = need(ns, \"cond_table\")(df, \"hava\", \"oyna\", alpha=1.0)\n        ensure(isinstance(t, pd.DataFrame), \"DataFrame bekleniyordu\")\n        close(t.loc[[\"E\", \"H\"], [\"g\", \"y\"]].to_numpy(dtype=float), [[0.6, 0.4], [0.4, 0.6]])\n        t0 = need(ns, \"cond_table\")(df, \"hava\", \"oyna\", alpha=0.0)\n        close(t0.loc[\"E\", \"g\"], 2 / 3, what=\"α = 0\")\n    R.test(\"cond_table: Laplace'lı koşullu olasılık tablosu\", tab)\n",
 "d": "import torch\nfrom labcheck import close, need, ensure\n\n\ndef spam(g, n=400, V=30):\n    y = torch.randint(0, 2, (n,), generator=g)\n    p0 = torch.softmax(torch.randn(V, generator=g), 0)\n    p1 = torch.softmax(torch.randn(V, generator=g), 0)\n    X = torch.stack([torch.multinomial(p1 if c else p0, 20, replacement=True, generator=g).bincount(minlength=V) for c in y]).double()\n    return X, y\n\n\ndef check(ns, R):\n    g = torch.Generator().manual_seed(0)\n    X, y = spam(g)\n\n    def fit():\n        lp, lt = need(ns, \"fit_mnb\")(X, y, 2, alpha=0.5)\n        ensure(tuple(lt.shape) == (2, 30), f\"log_theta shape {tuple(lt.shape)}\")\n        for c in range(2):\n            N = X[y == c].sum(0)\n            close(lt[c], torch.log((N + 0.5) / (N.sum() + 0.5 * 30)), what=f\"sınıf {c}\")\n            close(lp[c].exp(), (y == c).double().mean(), what=\"prior\")\n        close(lt.exp().sum(1), torch.ones(2, dtype=lt.dtype), what=\"θ satır toplamı\")\n    R.test(\"fit_mnb: Laplace'lı θ ve prior\", fit)\n\n    def pred():\n        m = need(ns, \"fit_mnb\")(X, y, 2)\n        L = need(ns, \"predict_mnb\")(m, X)\n        close(L.exp().sum(1), torch.ones(400, dtype=L.dtype), what=\"normalize\")\n        acc = (need(ns, \"predict_mnb\")(m, X).argmax(1) == y).double().mean().item()\n        ensure(acc > 0.85, f\"eğitim doğruluğu {acc:.2f}\")\n        return f\"eğitim doğruluğu {acc:.3f}\"\n    R.test(\"predict_mnb normalize, doğruluk > 0.85\", pred)\n",
 "e": "import torch\nimport torch.nn.functional as F\nfrom labcheck import close, no_loops, need, ensure, bench, fmt_time\n\n\ndef acc_one(Xtr, ytr, Xva, yva, C, a):\n    N = F.one_hot(ytr, C).to(Xtr.dtype).T @ Xtr\n    lt = torch.log(N + a) - torch.log(N.sum(1, keepdim=True) + a * Xtr.shape[1])\n    lp = torch.log(torch.bincount(ytr, minlength=C).to(Xtr.dtype) / len(ytr))\n    return ((Xva @ lt.T + lp).argmax(1) == yva).to(Xtr.dtype).mean()\n\n\ndef data(g, n, V, C):\n    y = torch.randint(0, C, (n,), generator=g)\n    P = torch.softmax(torch.randn(C, V, generator=g) * 1.5, 1)\n    X = torch.stack([torch.multinomial(P[c], 15, replacement=True, generator=g).bincount(minlength=V) for c in y]).double()\n    return X, y\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    g = torch.Generator().manual_seed(0)\n    Xtr, ytr = data(g, 300, 80, 3)\n    Xva, yva = data(g, 200, 80, 3)\n    alphas = torch.logspace(-4, 2, 20, dtype=torch.float64)\n\n    def eq():\n        A = need(ns, \"alpha_sweep\")(Xtr, ytr, Xva, yva, 3, alphas)\n        ensure(tuple(A.shape) == (20,), f\"shape {tuple(A.shape)}\")\n        ref = torch.stack([acc_one(Xtr, ytr, Xva, yva, 3, float(a)) for a in alphas])\n        close(A, ref)\n        k = int(A.argmax())\n        return f\"en iyi α = {float(alphas[k]):.3g}, doğrulama doğruluğu {float(A[k]):.3f}\"\n    R.test(\"20 α: tek tek eğitimle aynı\", eq)\n    if \"alpha_sweep\" not in ns:\n        return\n    Xb, yb = data(g, 2000, 500, 5)\n    al = torch.logspace(-4, 2, 20, dtype=torch.float64)\n    t_loop = bench(lambda: [acc_one(Xb, yb, Xb, yb, 5, float(a)) for a in al], repeat=3)\n    t_vec = bench(lambda: ns[\"alpha_sweep\"](Xb, yb, Xb, yb, 5, al), repeat=3)\n    R.table(\"20 α, n = 2000, V = 500\", [\"yöntem\", \"süre\", \"döngüye göre\"],\n            [[\"for α: eğit + tahmin\", fmt_time(t_loop), \"1×\"], [\"senin alpha_sweep (einsum)\", fmt_time(t_vec), f\"{t_loop / t_vec:,.1f}× hızlı\"]],\n            note=\"Sayım matrisi N tek kez hesaplanır; α yalnız logaritmaya giren bir sabit.\")\n"
}

In [ ]:
import importlib, labcheck, checks_m2_2_naive_bayes as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Bayes kuralı, sınıfın veriye göre olasılığını verinin sınıfa göre olasılığından çevirir:

$$
p(y\mid x)=\frac{p(y)\,p(x\mid y)}{p(x)}\;\propto\;p(y)\,p(x\mid y).
$$

$p(x\mid y)$'yi $d$ boyutlu $x$ için öğrenmek çok veri ister. **Naive** varsayım: sınıf bilindiğinde özellikler birbirinden bağımsız,

$$
p(y\mid x)\;\propto\;p(y)\prod_{j=1}^{d}p(x_j\mid y)
\qquad\Rightarrow\qquad
\log p(y\mid x)=\log p(y)+\sum_j\log p(x_j\mid y)+\text{sabit}.
$$

Her terim sayılarak (MLE, M0.4) tahmin edilir. Üç yaygın tür: **kategorik** (her özellik bir kategori), **Gaussian** (her özellik sınıfa göre bir normal dağılım) ve **multinomial** (kelime sayımları).

Bir değer bir sınıfta hiç görülmediyse MLE ona 0 verir ve çarpımın tamamı 0 olur. Laplace düzeltmesi her sayıma $\alpha$ ekler: $\hat p=\frac{n_{v,c}+\alpha}{N_c+\alpha V}$.

## 1 · Elle

Tabloyu iki sütunlu (E ve H) bir sayım tablosuna dönüştür, olasılıkları oku, sorgu için iki skoru hesapla ve toplamlarına bölerek normalize et.

Sonra (yağmur, zayıf) sorgusunu düşün: H sınıfında hiç zayıf rüzgâr yok. Düzeltmesiz model, diğer kanıt ne olursa olsun H'ye sıfır verir. Laplace ile tekrar hesapla; aynı sorguda posterior artık bir sayı.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Tablo (hava, rüzgâr → oyna): (güneş, zayıf, E), (güneş, güçlü, E), (yağmur, güçlü, H), (yağmur, zayıf, E), (güneş, güçlü, H), (yağmur, güçlü, H). Sorgu: (güneş, güçlü)**

- `prior`: $P(E)$
- `sunE`: $P(\text{güneş}\mid E)$
- `strE`: $P(\text{güçlü}\mid E)$
- `strH`: $P(\text{güçlü}\mid H)$
- `post`: $P(E\mid \text{güneş},\text{güçlü})$

**Sıfır sayım ve Laplace ($\alpha=1$, her özellik 2 değerli): $\frac{\#+\alpha}{N_c+2\alpha}$**

- `zero`: Düzeltmesiz $P(\text{zayıf}\mid H)$
- `lstrH`: Laplace $P(\text{güçlü}\mid H)$
- `lpost`: Laplace ile $P(E\mid\text{güneş},\text{güçlü})$

In [ ]:
cevaplar = dict(
    prior=None,
    sunE=None,
    strE=None,
    strH=None,
    post=None,
    zero=None,
    lstrH=None,
    lpost=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"prior":[0.5,0.001],"sunE":[0.666667,0.002],"strE":[0.333333,0.002],"strH":[1,0.001],"post":[0.4,0.002],"zero":[0,0.0001],"lstrH":[0.8,0.001],"lpost":[0.428571,0.002]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, kategorik Naive Bayes. Satırlar tuple, etiketler liste. `math` ve `collections` serbest.

- `fit_nb(rows, labels, alpha=1.0)` → bir model (yapısı senin seçimin). Laplace paydasındaki $V_j$, $j$. özelliğin eğitimde görülen **farklı** değer sayısı.
- `predict_proba(model, row)` → `{sınıf: olasılık}`; skorları **log uzayında** topla, en büyük log'u çıkararak normalize et.
- `predict(model, row)` → en olası sınıf.

Kontrol, `alpha=0` ile elle bulduğun 0.4'ü, `alpha=1` ile 0.4286'yı ve görülmemiş bir değerin sıfır olasılık üretmesini test eder.

In [ ]:
%%writefile stage_b.py
import math
from collections import Counter, defaultdict


def fit_nb(rows, labels, alpha=1.0):
    ...


def predict_proba(model, row):
    ...


def predict(model, row):
    ...


rows = [("g", "z"), ("g", "s"), ("y", "s"), ("y", "z"), ("g", "s"), ("y", "s")]
labels = ["E", "E", "H", "E", "H", "H"]
print(predict_proba(fit_nb(rows, labels, alpha=0.0), ("g", "s")))   # E: 0.4
print(predict_proba(fit_nb(rows, labels, alpha=1.0), ("g", "s")))   # E: 0.4286

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

İki parça, ikisi de **döngüsüz**.

**Gaussian NB (NumPy).** Her sınıf $c$ ve özellik $j$ için $\mu_{cj}$ ve $\sigma^2_{cj}$ (MLE). Sınıf istatistiklerini döngü yerine one-hot matrisle al: `Y = np.eye(C)[y]` `(n, C)`, sayımlar `Y.sum(0)`, ortalamalar `Y.T @ X / sayım[:, None]`.

- `fit_gnb(X, y, C, eps=1e-9)` → `(log_prior: (C,), mu: (C, d), var: (C, d))`; varyansa `eps` ekle.
- `predict_log_proba_gnb(model, X)` → `(n, C)` normalize log-posterior. Log-yoğunluk için `X[:, None, :]` ile `mu[None]` broadcasting ile `(n, C, d)`.

**Olasılık tablosu (pandas).**

- `cond_table(df, feature, target, alpha=1.0)` → satırları `target` sınıfları, sütunları `feature` değerleri olan $P(\text{değer}\mid\text{sınıf})$ DataFrame'i (Laplace'lı). `pd.crosstab` tek satır.

In [ ]:
%%writefile stage_c.py
import numpy as np
import pandas as pd


def fit_gnb(X, y, C, eps=1e-9):
    ...


def predict_log_proba_gnb(model, X):
    ...


def cond_table(df, feature, target, alpha=1.0):
    ...


rng = np.random.default_rng(0)
y = np.repeat([0, 1], 50)
X = rng.normal(size=(100, 2)) + np.array([[0, 0], [2, 2]])[y]
m = fit_gnb(X, y, 2)
print((predict_log_proba_gnb(m, X).argmax(1) == y).mean())
df = pd.DataFrame({"hava": list("ggyygy"), "oyna": list("EEHEHH")})
print(cond_table(df, "hava", "oyna"))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Metin sınıflandırması: her belge bir kelime sayım vektörü $x\in\mathbb N^V$. Multinomial NB:

$$
\log p(y=c\mid x)=\log\pi_c+\sum_v x_v\log\theta_{cv}+\text{sabit},\qquad
\theta_{cv}=\frac{N_{cv}+\alpha}{\sum_{v'}N_{cv'}+\alpha V}.
$$

$N_{cv}$, $c$ sınıfındaki belgelerde $v$ kelimesinin toplam sayısı: `F.one_hot(y, C).T.float() @ X` `(C, V)`.

- `fit_mnb(X, y, C, alpha=1.0)` → `(log_prior: (C,), log_theta: (C, V))`.
- `predict_mnb(model, X)` → `(n, C)` normalize log-olasılık; `torch.logsumexp` ile normalize et.

Kontrol, küçük sentetik bir "spam" veri setinde doğruluğa ve elle yazılmış referansa bakar.

In [ ]:
%%writefile stage_d.py
import torch
import torch.nn.functional as F


def fit_mnb(X, y, C, alpha=1.0):
    """X: (n, V) sayımlar → (log_prior: (C,), log_theta: (C, V))"""
    ...


def predict_mnb(model, X):
    """→ (n, C) normalize log-olasılık"""
    ...


X = torch.tensor([[3, 0, 1, 0], [2, 0, 0, 1], [0, 2, 0, 3], [0, 3, 1, 2]], dtype=torch.float32)
y = torch.tensor([0, 0, 1, 1])
print(predict_mnb(fit_mnb(X, y, 2), X).exp())

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Laplace katsayısı $\alpha$ bir hiperparametre. 20 değeri döngüyle denemek yerine tüm modelleri tek tensörde kur:

$$
\log\theta^{(a)}_{cv}=\log(N_{cv}+\alpha_a)-\log\Big(\sum_{v'}N_{cv'}+\alpha_aV\Big)\in\mathbb R^{A\times C\times V},
$$

doğrulama skorları `torch.einsum("nv,acv->anc", Xva, log_theta)` `(A, n, C)`.

- `alpha_sweep(Xtr, ytr, Xva, yva, C, alphas)` → doğrulama doğrulukları `(A,)`. **Döngü yok.**

Kontrol sonucu her α için ayrı eğitimle karşılaştırır ve hız tablosu üretir.

In [ ]:
%%writefile stage_e.py
import torch
import torch.nn.functional as F


def alpha_sweep(Xtr, ytr, Xva, yva, C, alphas):
    """→ (A,) doğrulama doğrulukları. Döngü yok."""
    ...


g = torch.Generator().manual_seed(0)
Xtr = torch.randint(0, 3, (200, 50), generator=g).float()
ytr = torch.randint(0, 2, (200,), generator=g)
print(alpha_sweep(Xtr, ytr, Xtr, ytr, 2, torch.logspace(-3, 1, 5)))

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Kaynaklar

- [Stanford CS229 — Lecture notes: Generative learning algorithms, Naive Bayes](https://cs229.stanford.edu/main_notes.pdf) — doğrulanmadı
- [Murphy — PML1 §9.3 Naive Bayes classifiers](https://probml.github.io/pml-book/book1.html) — CC BY-NC-ND
- [scikit-learn User Guide — Naive Bayes](https://scikit-learn.org/stable/modules/naive_bayes.html) — BSD-3
- [Manning, Raghavan, Schütze — IR Book, Bölüm 13 (multinomial NB)](https://nlp.stanford.edu/IR-book/html/htmledition/naive-bayes-text-classification-1.html) — ©